## Targeted Sentiment & PII Detection

### 학습 목표
1. `detect_targeted_sentiment`로 개체별 세밀한 감성을 분석한다.
2. `detect_pii_entities`로 개인식별정보(PII)를 자동 감지한다.
3. PII를 마스킹 처리하는 함수를 구현한다.

### API 개요
```python
# 대상 감성 분석 (개체별 감성)
comprehend.detect_targeted_sentiment(Text='...', LanguageCode='en')

# PII 감지
comprehend.detect_pii_entities(Text='...', LanguageCode='en')
```

### Targeted Sentiment 결과 구조
```
{
  'Text': 'battery life',
  'Type': 'ATTRIBUTE',
  'Mentions': [{
    'Text': 'battery life',
    'MentionSentiment': {'Sentiment': 'POSITIVE', 'SentimentScore': {...}}
  }]
}
```

> ⚠️ detect_targeted_sentiment 및 detect_pii_entities는 **영어(en)**만 지원함.


In [ ]:
#환경 초기화
import boto3
import matplotlib.pyplot as plt
import pandas as pd
import re

# 한글 폰트 설정 (SageMaker Studio)
try:
    import koreanize_matplotlib
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib

#대상 감성(targeted_sentiment)·PII 감지는 영어(en)만 지원 → us-east-1 사용
comprehend = boto3.client('comprehend', region_name='us-east-1')
print('Comprehend 클라이언트 생성 완료 (us-east-1)')


In [ ]:
#detect_targeted_sentiment

review = (
    "The camera quality is absolutely stunning and takes great photos. "
    "However, the battery life is terrible and drains too fast. "
    "The design looks premium but the screen has some issues."
)

response = comprehend.detect_targeted_sentiment(
    Text=review,         
    LanguageCode='en'   
)

entities = response['Entities']
print(f'분석된 개체: {len(entities)}개')
print('-' * 60)
for e in entities:
    mention = e['Mentions'][0]
    text     = mention['Text']        
    sent     = mention['MentionSentiment']['Sentiment']  
    score    = mention['MentionSentiment']['SentimentScore']
    dominant = max(score, key=score.get)
    print(f"  {text:<20} → {sent:<12} (최고: {dominant} {score[dominant]:.3f})")


In [ ]:
# 개체별 감성 비교 시각화
rows = []
for e in entities:
    m    = e['Mentions'][0]
    sc   = m['MentionSentiment']['SentimentScore']
    rows.append({'entity': m['Text'], 'sentiment': m['MentionSentiment']['Sentiment'],
                 'pos': sc['Positive'], 'neg': sc['Negative']})
df = pd.DataFrame(rows).drop_duplicates('entity')

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(df))
ax.bar([i-0.2 for i in x], df['pos'], 0.35, label='Positive', color='#2ecc71')
ax.bar([i+0.2 for i in x], df['neg'], 0.35, label='Negative', color='#e74c3c')
ax.set_xticks(list(x))
ax.set_xticklabels(df['entity'], rotation=20, ha='right')
ax.set_title('개체별 감성 점수 (Targeted Sentiment)')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# detect_pii_entities

pii_text = (
    "Hello, my name is John Smith. You can reach me at john.smith@email.com "
    "or call me at 555-123-4567. My address is 123 Main Street, New York, NY 10001. "
    "My credit card number is 4532-1234-5678-9012."
)

response = comprehend.detect_pii_entities(
    Text=pii_text, 
    LanguageCode='en'
)

pii_entities = response['Entities']
print(f'감지된 PII: {len(pii_entities)}개')
print('-' * 60)
for p_ent in pii_entities:
    start = p_ent['BeginOffset']   
    end   = p_ent['EndOffset']  
    text  = pii_text[start:end]
    print(f"  [{p_ent['Type']:<20}] {text:<30} (신뢰도: {p_ent['Score']:.3f})")


In [ ]:
# PII 마스킹 함수 구현

def mask_pii(text, pii_entities):
    """PII 위치를 뒤에서부터 치환하여 offset 오염 방지"""
    sorted_entities = sorted(pii_entities,
                              key=lambda x: x['BeginOffset'], 
                              reverse=True)  
    masked = text
    for ent in sorted_entities:
        start  = ent['BeginOffset'] 
        end    = ent['EndOffset']  
        marker = f"[{ent['Type']}]"
        masked = masked[:start] + marker + masked[end:]
    return masked

masked_text = mask_pii(pii_text, pii_entities)
print('원본:')
print(' ', pii_text)
print('\n마스킹 결과:')
print(' ', masked_text)
